# 00 — Protocol and governance gate

This notebook freezes the computational context and reports the submission
governance gate before any result is interpreted. The primary estimand is
post-development, within-dataset cross-validated performance on held-out
**source videos**. A source video is
the independent unit; persistent person identifiers are unavailable, so the
workflow cannot support an unseen-person claim. Folder names are dataset
annotations, not diagnoses, and no diagnostic or clinical claim is made.

The protocol separates three claims that must not be collapsed: an analytic
wrapper can force odd output; an anatomy-aware encoder-plus-probe can exhibit
useful native odd behavior; and a checkpoint can satisfy a direct strict
token-equivariance test. Only the last two are empirical, and neither removes
the BlazePose schema, preprocessing, architecture, or source-sampling
assumptions supplied to the experiment.

The checked-in governance record intentionally fails closed until an
institutional ethics determination, a data-use review, and a derived-pose
release review each have a dated internal reference. Public availability is
not a substitute for those determinations. This suite never redistributes
raw video or identity-bearing frames. Linkable identifiers, derived poses,
embeddings, and checkpoints may be released only as completed reviews permit.

## What this notebook is testing

Notebook 00 does not test a model. It tests whether the project is about to run
under the intended computational rules and whether its submission/release gate
is administratively complete. Think of it as the label on a sealed experiment:
it states the question, the unit being analyzed, the selected model recipes,
and the rules that will later decide what can be claimed.

An **estimand** is the quantity the study intends to estimate. Here it is
post-development, within-dataset cross-validated performance on held-out source
videos from the Gait Abnormality in Video Dataset (GAVD). "Within-dataset"
means the result stays inside GAVD; "held-out" means a source video is evaluated
by a model that did not train on that source.
It does not mean that a person is known to be absent from every other video,
because persistent person identifiers are unavailable.

The protocol also separates three left–right ideas. An **odd output** obeys
$f(Mx)=-f(x)$ after input $x$ is anatomically mirrored by $M$; a mathematical
wrapper can force this property. **Native probe behavior** asks whether an
unconstrained read-out already behaves usefully. **Representation
equivariance** asks the stricter question of whether internal tokens themselves
transform according to the registered anatomical joint swap. Later notebooks
test these separately so success on an engineered wrapper cannot be described
as symmetry learned by the encoder.

The governance gate uses logical **AND**, not majority vote. Every review must
be marked `resolved` and must include both an approved internal reference and a
date. `Unresolved` means that the repository has no recorded determination; it
does not mean “rejected,” and it must not be guessed from public availability.

<svg viewBox="0 0 980 230" width="100%" role="img"
     aria-labelledby="governance-flow-title governance-flow-description"
     xmlns="http://www.w3.org/2000/svg">
  <title id="governance-flow-title">Submission governance AND gate</title>
  <desc id="governance-flow-description">Ethics, data-use, and derived-pose
  release reviews must all be resolved with a reference and date before the
  submission and release gate is ready.</desc>
  <defs><marker id="arrow00" markerWidth="8" markerHeight="8" refX="7"
    refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#475569"/></marker></defs>
  <style>
    .review00{fill:#fff1f2;stroke:#be123c;stroke-width:1.5}
    .gate00{fill:#f8fafc;stroke:#334155;stroke-width:1.5}
    .blocked00{fill:#fee2e2;stroke:#991b1b;stroke-width:2}
    .line00{stroke:#475569;stroke-width:1.8;fill:none;marker-end:url(#arrow00)}
    .h00{font:600 15px system-ui,sans-serif;fill:#0f172a}
    .s00{font:12px system-ui,sans-serif;fill:#475569}
    .r00{font:600 12px system-ui,sans-serif;fill:#9f1239}
  </style>
  <rect class="review00" x="20" y="15" width="255" height="55" rx="9"/>
  <rect class="review00" x="20" y="87" width="255" height="55" rx="9"/>
  <rect class="review00" x="20" y="159" width="255" height="55" rx="9"/>
  <text class="h00" x="147" y="38" text-anchor="middle">Ethics determination</text>
  <text class="r00" x="147" y="57" text-anchor="middle">currently unresolved</text>
  <text class="h00" x="147" y="110" text-anchor="middle">Data-use review</text>
  <text class="r00" x="147" y="129" text-anchor="middle">currently unresolved</text>
  <text class="h00" x="147" y="182" text-anchor="middle">Derived-pose release review</text>
  <text class="r00" x="147" y="201" text-anchor="middle">currently unresolved</text>
  <rect class="gate00" x="370" y="75" width="180" height="80" rx="10"/>
  <text class="h00" x="460" y="105" text-anchor="middle">All three complete?</text>
  <text class="s00" x="460" y="128" text-anchor="middle">status + reference + date</text>
  <text class="h00" x="460" y="148" text-anchor="middle">AND</text>
  <path class="line00" d="M275 42 C330 42 330 95 370 95"/>
  <path class="line00" d="M275 114 L370 114"/>
  <path class="line00" d="M275 186 C330 186 330 135 370 135"/>
  <rect class="blocked00" x="650" y="75" width="290" height="80" rx="10"/>
  <text class="h00" x="795" y="105" text-anchor="middle">Submission/release readiness</text>
  <text class="r00" x="795" y="132" text-anchor="middle">BLOCKED until every input is complete</text>
  <path class="line00" d="M550 115 L650 115"/>
</svg>

## How to read the protocol and governance progress display

This audit has three short stages: validate the governance record, assemble the
protocol snapshot, and render the status-only figure. It normally finishes in
seconds. The bar makes the active check explicit and turns an exception into a
visible failed stage. Because the stages are brief and differently sized, its ETA
is only a rough indication; `estimating…` is normal at the beginning.

A completed progress bar means the files were read and the checks ran. It does
not mean that the governance gate passed. The separate red or green governance
figure remains authoritative about readiness.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display


def locate_suite_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for ancestor in (start, *start.parents):
        for candidate in (ancestor, ancestor / "neurips-laterality"):
            if (
                (candidate / "config" / "protocol.json").is_file()
                and (candidate / "laterality").is_dir()
            ):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate neurips-laterality from the current working directory."
    )


SUITE_ROOT = locate_suite_root()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))

from laterality.config import load_context

context = load_context(SUITE_ROOT / "config" / "protocol.json")
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")


def show_inline(figure):
    display(figure)
    plt.close(figure)


print(
    f"suite={SUITE_ROOT} profile={context.profile} "
    f"artifacts={context.artifact_root} protocol={context.protocol_digest[:12]}"
)

In [ ]:
from laterality.config import model_config
from laterality.governance import load_governance, submission_readiness
from laterality.visualization import governance_figure
from notebook_progress import NotebookTaskProgress

governance_progress = NotebookTaskProgress(
    "Protocol and governance audit progress",
    "stage",
    refresh_seconds=0.25,
)
governance_progress.start(3, profile=context.profile)

with governance_progress.unit(1, "Load and validate governance record"):
    governance_path = SUITE_ROOT / "governance" / "status.json"
    governance_payload = load_governance(governance_path)
    governance_readiness = submission_readiness(governance_payload)

with governance_progress.unit(2, "Assemble the frozen protocol snapshot"):
    protocol_snapshot = {
        "profile": context.profile,
        "synthetic_smoke": not context.is_paper,
        "protocol_digest": context.protocol_digest,
        "claim": context.protocol["claim_boundary"]["primary"],
        "unsupported_claims": context.protocol["claim_boundary"]["not_supported"],
        "independent_unit": context.protocol["claim_boundary"]["independent_unit"],
        "selected_folds": list(context.folds),
        "selected_seeds": list(context.seeds),
        "selected_variants": list(context.variants),
        "model": model_config(context),
        "primary_lane": context.protocol["evaluation"]["primary_lane"],
        "constructed_repair_lane": context.protocol["evaluation"][
            "constructed_repair_lane"
        ],
        "primary_seed_estimand": context.protocol["evaluation"][
            "primary_seed_estimand"
        ],
        "representation_equivariance": context.protocol["evaluation"][
            "representation_equivariance"
        ],
        "decision_rules": context.protocol["evaluation"]["decision_rules"],
    }

with governance_progress.unit(3, "Render the status-only governance figure"):
    show_inline(governance_figure(context, governance_payload))
governance_progress.complete(status="Protocol and governance audit complete")
protocol_snapshot, governance_readiness

## How to interpret the current output

**Read the red bars as gate states, not scientific failures.** The checked-in
record currently marks the ethics determination, data-use review, and
derived-pose release review as `unresolved`; their reference and date fields
are empty. `governance_readiness["ready"]` is therefore `False`, and all three
names appear in its `unresolved` list. The plot suppresses references by design
and shows only whether each required record is complete.

This means the repository does not currently authorize a claim that the work
is submission-ready or that poses, embeddings, or checkpoints may be released.
It does not itself determine whether a particular institution permits local
analysis while reviews are pending; that question must follow the applicable
institutional process. Never change a status to make the bar green unless the
determination actually exists. Record only the approved internal reference and
date—do not copy confidential review documents into the repository.

**Then read the protocol snapshot as a contract, not a result.** For the paper
profile it should identify all five outer folds, seeds 42–46, and the two
variants `vanilla` and `reflection_augmented`. The model snapshot specifies 64
frames, 33 landmarks, three coordinates, four-frame temporal patches, a
96-dimensional embedding, four encoder layers, two predictor layers, and four
attention heads. These values tell later artifacts what implementation they
must match; they do not say that this model performs well.

The primary lane `learned_single_free` means a learned encoder, one original
input pass, and an unconstrained linear read-out. The constructed repair lane
`learned_two_pass_odd_zero` combines original and mirrored representations to
force odd features and uses a zero-origin read-out. Exact oddness in that lane
proves the construction, not that the unconstrained encoder learned symmetry.
Likewise, the registered $R^2$ and equivariance margins are future decision
rules, not scores already achieved.

SHA stands for Secure Hash Algorithm; SHA-256 produces the 256-bit content
fingerprint used as the protocol digest. The displayed prefix should begin
`6f7baefbda07` for the current
locked protocol. A different digest is not automatically wrong, but it means
results belong to a different protocol and must not be mixed with these
artifacts.

The narrow conclusion from notebook 00 is: **the paper computation is precisely
specified, but the current submission/release governance gate is not complete.**
This remains true even if every later model finishes successfully.

A `smoke` run uses generated poses and a tiny model only to test plumbing,
lineage checks, and algebraic invariants. Its metrics are **not empirical
evidence** and must never enter a paper table. Likewise, a completed paper run
does not override an unresolved governance gate.